# Week 07: Attention, GPT and applications

## Text as Data

Professor: Elliott Ash, NYU

TA: Eduardo Zago, NYU

Last class we went over (finalizing) pre-processing of text into tokens, embeddings (context and positional) and into a basic attention mechanism.

Objectives:

1) Enhanced Self-Attention
2) Causal Self-Attention
3) Multi-head Attention
4) All together: GPT-2
5) Apps: BERTopic and DeepLatent

Code adapted from Raschka Ch: 3.4-3.6

Also see:



* Andrej Karpathy's [micro-gpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)
*  An [explanation](https://www.towardsdeeplearning.com/andrej-karpathy-just-built-an-entire-gpt-in-243-lines-of-python-7d66cfdfa301)
* And this [blog](https://jaykmody.com/blog/gpt-from-scratch)









In [1]:
# set random seed
!pip install torch --upgrade
!pip install gensim
!pip install tiktoken

import collections
from collections import Counter

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.optim as optim
from tqdm.notebook import tqdm
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from string import punctuation
nltk.download('stopwords')
from nltk.corpus import stopwords
stoplist = set(stopwords.words('english'))

from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.8 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Last class we saw the simple attention mechanism. Main idea: for each query, we need to understand how the other tokens in the context window relate to it. We do this by computing similarities, normalizing and weighting the initial outputs with respect to a certain query. Steps:

1) For $j\leq i$, compute $sim(x_i, x_j) = x_i^{T}x_j$
2) Normalize to get attention weights: $\alpha_{ij} = (softmax([sim(x_i, x_1), ..., sim(x_i, x_j)])_j$
3) Compute attention scores for each query: $a_i = \sum_{j \leq i} \alpha_{ij}x_j$


In [4]:
# Keep the first sentence:

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = attn_weights @ inputs

print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Enhanced self-attention

Notice that the same word can play three different roles in this process:

1) When it is the current word $i$ in calculating $sim(x_i, x_j)$.
2) When it is surrounding word $j$ in calculating $sim(x_i, x_j)$
3) When it is surrounding word $j$ in calculcating $\sum_{j \leq i} \alpha_{ij}x_j$

Thus we will use three different vector to represent each word in a different role, and add parameters that our network can learn:

1) query ($q_i = W_q x_i$)
2) key ($k_i = W_k x_i$)
3) value ($v_i = W_v x_i$).

In [6]:
x_2 = inputs[1]  # As last exercise, second token is the query
d_in = inputs.shape[1]  # Input embedding size
d_out = 2  # output (different from input for didactic reasons)

# Initialize the weight (parameter) matrices:

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# Compute query, key and value vectors

query_2 = x_2 @ W_query # current item
key_2 = x_2 @ W_key # indexing and searching
value_2 = x_2 @ W_value # actual content
print(query_2)

tensor([0.4306, 1.4551])


Compute similarities as we did before, with respect to the query:

$$sim(x_i, x_j) = \frac{q_i^{T} k_j}{\sqrt{d_k}}$$

In [13]:
# We need al keys and values weights to compute the attention score of the second input

keys = inputs @ W_key
values = inputs @ W_value

d_k = keys.shape[-1]

attn_scores_2 = query_2 @ keys.T / d_k**0.5  # scaled dot product (to avoid backpropagation issues, such as really small gradients)
print(attn_scores_2)

tensor([0.8984, 1.3098, 1.2806, 0.7633, 0.3944, 1.0918])


Normalize them:

$\alpha_{ij} = (softmax([sim(x_i, x_1), ..., sim(x_i, x_j)])_j$

In [14]:
# Attention scores to attention weights:

attn_weights_2 = torch.softmax(attn_scores_2, dim=-1) #
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Compute the final context vector:

$\sum_{j \leq i} \alpha_{ij}v_j$

In [15]:
# Context vectors:

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


Compacting it all:

In [21]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # Linear with bias = False == matrix mult with opt. weight initialization
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v1(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


### Causal attention mechanism

For decoder tasks, we will want the self-attention mechanisms to only consider tokens that appear prior to the current position.

In [23]:
queries = sa_v2.W_query(inputs)     #1
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

context_length = attn_scores.shape[0]
###

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

###

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)
tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000

In [25]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


batch = torch.stack((inputs, inputs), dim=0)
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                 d_in, d_out, context_length, dropout, qkv_bias
             ) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

Next week: GPT2 IMPLEMENTATION!! (Finally).

For now we can look at some applications of the embeddings we have generated, as well as other tools we have used before:

### BERTOpic

<div style="text-align: center;">
<img src='https://maartengr.github.io/BERTopic/img/algorithm.png' width='600'>
</div>

In [2]:
!pip install bertopic[visualization]
from bertopic import BERTopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [3]:
from google.colab import files
uploaded_2 = files.upload()

Saving death-penalty-cases.csv to death-penalty-cases.csv


In [22]:
df1 = pd.read_csv('death-penalty-cases.csv')
docs = df1['snippet'].tolist()
len(docs)

32567

In [5]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(docs[:400])

# Explore topics
print(topic_model.get_topic_info().head(10))

2026-03-08 22:25:15,882 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1018 [00:00<?, ?it/s]

2026-03-08 22:25:53,732 - BERTopic - Embedding - Completed ✓
2026-03-08 22:25:53,737 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-08 22:26:47,682 - BERTopic - Dimensionality - Completed ✓
2026-03-08 22:26:47,684 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-08 22:34:38,995 - BERTopic - Cluster - Completed ✓
2026-03-08 22:34:39,013 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-08 22:34:40,234 - BERTopic - Representation - Completed ✓


   Topic  Count                                               Name  \
0     -1  13230                            -1_the_to_death_penalty   
1      0    624                  0_appellant_appellants_guilty_was   
2      1    490               1_published_nondeath_felony_november   
3      2    423                         2_texas_scheme_error_point   
4      3    406                         3_notice_intent_filed_seek   
5      4    381                  4_hearing_eligible_preclude_found   
6      5    366        5_florida_unconstitutional_statute_floridas   
7      6    346           6_sanctions_sanction_discovery_quotdeath   
8      7    320  7_unconstitutional_constitutionality_supreme_s...   
9      8    316              8_petitioner_petitioners_he_testified   

                                      Representation  \
0  [the, to, death, penalty, that, not, in, he, w...   
1  [appellant, appellants, guilty, was, assessed,...   
2  [published, nondeath, felony, november, octobe...   
3  [t

In [6]:
# Visualizations:
topic_model.visualize_topics()                        # intertopic distance map

In [7]:
topic_model.visualize_barchart(top_n_topics=8)        # top words per topic

In [8]:
topic_model.visualize_heatmap()

In [10]:
# ── 5. Assign topics back to dataframe ───────────────────────────────────────
df1['topic'] = topics
df1['topic_label'] = df1['topic'].apply(
    lambda t: " | ".join([w for w, _ in topic_model.get_topic(t)[:3]]) if t != -1 else "outlier"
)
print(df1[['snippet', 'topic', 'topic_label']].head())

                                             snippet  topic  \
0  N.J.   ( )\n  A. d  \nIN RE WAIVER OF DEATH PE...    102   
1  whether the death penalty is, per se, unconsti...      5   
2  # ;s contention that the assessment of the dea...     -1   
3  . d   ( )\n -NMSC- \nIN THE MATTER OF DEATH PE...     -1   
4  assume the district attorney orally waived the...     27   

                            topic_label  
0              jersey | new | jerseyans  
1  florida | unconstitutional | statute  
2                               outlier  
3                               outlier  
4           state | seeking | agreement  


### DeepLatent:

Based on: [DeepLatent](https://github.com/PinchOfData/DeepLatent). Topic modelling is a way of extracting latent topics from unstructured text. However, there are a few things to have in mind while doing this task:

What are covariates in a topic model?
Two types:
1) prevalence: a covariate that affects HOW MUCH each topic appears in a doc   e.g. "does the year of the case change which topics dominate?"

2) content:    a covariate that affects THE WORDS used to discuss a topic e.g. "do defense lawyers and prosecutors use different words for the same topic?"

Example: imagine two death-penalty cases about "mental illness".

prevalence covariate → older cases talk less about mental illness overall content covariate    → prosecution says "dangerous", defense says "disorder"

In [11]:
!pip install deeplatent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.6 MB/s eta 0:00:00


In [19]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from deeplatent.corpus import Corpus
from deeplatent.models import GTM


In [23]:
# 1. Define and fit the vectorizer using training data
df1 = df1.sample(400)

docs = df1['snippet'].tolist()

vectorizer = CountVectorizer(
    ngram_range=(1, 1),
    max_df=0.95,
    min_df=0.001,
    stop_words="english",
    token_pattern=r"(?u)\b\w{3,}\b"  # only words with 3 or more letters
)

vectorizer.fit(docs)

print("Number of words in the vocabulary: {}".format(len(vectorizer.get_feature_names_out())))

# 2. Define modalities using the external vectorizer
modalities = {
    "text": {
        "column": "snippet",
        "views": {
            "bow": {
                "type": "bow",
                "vectorizer": vectorizer
            }
        }
    }
}

# 3. Create GTMCorpus with prevalence and content covariates
# Setting to None simply trains a Wasserstein topic model (without metadata).
train_dataset = Corpus(
    df=df1,
    modalities=modalities,
    prevalence="~ C(year)",
    content="~ C(state)"
)

Number of words in the vocabulary: 1938


In [25]:
from deeplatent import GTM

encoder_args = {
    "text_bow": {
        "hidden_dims": [256,128],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

decoder_args = {
    "text_bow": {
        "hidden_dims": [128,256],
        "activation": "relu",
        "bias": True,
        "dropout": 0.1
    }
}

tm = GTM(
    train_dataset,
    n_topics=20,
    ae_type="wae", # WAE has very stable training and can use a Dirichlet prior, VAE has stronger theoretical guarantees but is prone to posterior collapse
    doc_topic_prior="logistic_normal",
    update_prior=True,
    encoder_args=encoder_args,
    decoder_args=decoder_args,
    batch_size=64,
    w_prior=1,
    return_best_model=True,
    num_steps=2000,
    print_every_n_steps=1000,
    print_topics=True,
    print_covariates=True
)

# Resume training if you think the topics need more refining
# tm.num_steps = tm.steps + 5000  # Train 5000 more steps
# tm.train(train_data)

Step   1000	Mean Training Loss:4.6626825
Rec Loss:4.4627895
Divergence Loss:0.1998930
Pred Loss:0.0000000

Topic_0: ['west', 'state', 'penitentiary', 'cases', 'facing']
Topic_1: ['case', 'murder', 'state', 'felony', 'defendant']
Topic_2: ['felony', 'murder', 'seeking', 'degree', 'ann']
Topic_3: ['quot', 'act', 'effective', 'antiterrorism', 'cases']
Topic_4: ['court', 'imposed', 'quot', 'murder', 'circumstances']
Topic_5: ['pub', 'act', 'effective', 'antiterrorism', 'quot']
Topic_6: ['antiterrorism', 'effective', 'aedpa', 'act', 'quot']
Topic_7: ['life', 'imprisonment', 'state', 'penitentiary', 'receive']
Topic_8: ['murder', 'contends', 'state', 'defendant', 'court']
Topic_9: ['quot', 'imposed', 'defendants', 'cases', 'shall']
Topic_10: ['quot', 'imposed', 'case', 'jury', 'cases']
Topic_11: ['seek', 'plea', 'trial', 'aggravating', 'entered']
Topic_12: ['statute', 'aggravating', 'imposition', 'circumstances', 'court']
Topic_13: ['jurors', 'trial', 'jury', 'circumstances', 'imposing']
Top

In [26]:
print(
    "\n".join(
        [
            "{}: {}".format(str(k), str(v))
            for k, v in tm.get_topic_words(topK=5).items()
        ]
    )
)

Topic_0: ['cases', 'facing', 'state', 'indictment', 'make']
Topic_1: ['state', 'case', 'murder', 'non', 'defendant']
Topic_2: ['murder', 'seeking', 'trial', 'felony', 'published']
Topic_3: ['verdict', 'imposed', 'quot', 'effective', 'act']
Topic_4: ['quot', 'court', 'state', 'punishment', 'imposed']
Topic_5: ['act', 'effective', 'antiterrorism', 'pub', 'quot']
Topic_6: ['effective', 'aedpa', 'antiterrorism', 'act', 'quot']
Topic_7: ['life', 'receive', 'did', 'petitioner', 'imprisonment']
Topic_8: ['state', 'impose', 'case', 'phase', 'court']
Topic_9: ['imposed', 'jury', 'quot', 'cases', 'case']
Topic_10: ['quot', 'case', 'cases', 'eligible', 'jury']
Topic_11: ['seek', 'did', 'aggravating', 'state', 'intent']
Topic_12: ['statute', 'murder', 'imposition', 'circumstances', 'cases']
Topic_13: ['jurors', 'jury', 'trial', 'state', 'court']
Topic_14: ['state', 'evidence', 'statement', 'court', 'circumstances']
Topic_15: ['state', 'proceedings', 'seeking', 'seek', 'trial']
Topic_16: ['quot', '

In [27]:
print("\nTop words per topic for a California case:")
print(pd.DataFrame(tm.get_topic_words(l_content_covariates = ["Intercept", "C(state)[T.CA]"], topK=5)).T)


Top words per topic for a California case:
                    0           1              2            3              4
Topic_0   superceding   exclusive        strikes   recidivist        schemes
Topic_1           law      murder        special  legislature      defendant
Topic_2       resting        bail        seeking       stated       admitted
Topic_3          quot   effective           case          act          aedpa
Topic_4          quot        case         suffer   discussion            use
Topic_5          quot   effective          added        aedpa  antiterrorism
Topic_6          quot       aedpa      effective          act  antiterrorism
Topic_7   description     morales         delays       refers         office
Topic_8           law        life         stated    supported    entertained
Topic_9          quot        case         future     pursuant          terms
Topic_10         quot   involving       sentence    automatic           case
Topic_11      persons  concernin